In [ ]:
import numpy as np
import scipy.sparse as sp
import time
from dolphindes.cvxopt import DenseSharedProjQCQP, OptimizationHyperparameters

# conda clean --all

In [ ]:
# Physical constants
epsilon_0 = 8.854e-12 # F/m
mu_0 = 4e-7 * np.pi # H/m
c = 1/np.sqrt(epsilon_0 * mu_0) # speed of light in vacuum

E0 = 1e3 # plane wave amplitude in V/m

frequency = 2.45e9 # frequency in Hz
wavelength = c / frequency # wavelength in m
print(f"Wavelength: {wavelength} m")

ka = 1 # electrical size of the antenna
k = 2 * np.pi / wavelength # wavenumber
a = ka / k # antenna circumradius
print(f"Antenna circumradius: {a} m")

omega = 2 * np.pi * frequency # angular frequency

conductivity_reduction_factor = 1 # factor to reduce conductivity for testing purposes
copper_conductivity = conductivity_reduction_factor*5.96e7 # S/m
copper_permittivity = 1 + 1j * copper_conductivity / (omega * epsilon_0) # copper permittivity in dimensionless units
print(f"Copper permittivity: {copper_permittivity:.2e}")

Wavelength: 0.1223655664053944 m
Antenna circumradius: 0.019475084757658086 m
Copper permittivity: 1.00e+00+4.37e+08j


In [ ]:
# Calculate surface impedance
delta = np.sqrt(2 / (omega * mu_0 * copper_conductivity)) # skin depth in m
Zs = (1 + 1j) / (copper_conductivity * delta)
# Zs = 1 / (copper_conductivity * delta)
print(f"Surface impedance: {Zs} Ω")

# Load matrices from text files
Lmat = np.loadtxt(r"Lmat_matrix.txt", delimiter=',') # lossy matrix
R0 = np.loadtxt(r"R0_matrix.txt", delimiter=',') # radiated power matrix
X0 = np.loadtxt(r"X0_matrix.txt", delimiter=',') # reactive power matrix
Vinc = E0*np.loadtxt(r"V_vector.txt", delimiter=',') # voltage excitation vector
Zmat = Zs * Lmat # material impedance matrix
Z0 = R0 + 1j*X0 # free-space impedance matrix
Ztot = Z0 + Zmat # total impedance matrix
Rmat = np.real(Zmat) # rezistivity matrix

Ndes = Lmat.shape[0] # number of design variables
print("Number of design variables: ", Ndes)

Vzero = np.zeros((Ndes,), dtype=complex) # zero vector

Obj = -0.5 * R0
obj = Vzero/2
obj0 = 0

# full plate performance
Ifull = np.linalg.solve(Ztot, Vinc) # current distribution for full plate
Pfull = -np.real(Ifull.conj().T @ Obj @ Ifull) + 2*np.real(Ifull.conj().T @ obj) + obj0 # objective for full plate
print(f"Objective for full plate: {Pfull:.6e} W")

Surface impedance: (0.012739130327240646+0.012739130327240646j) Ω
Number of design variables:  207
Objective for full plate: 7.046531e+00 W


In [ ]:
# # use Schur complemet to satisfy given iBFfixed row of system equation Ztot @ Ivec = Vinc
# iBFfixed = 261
# # Split the total impedance matrix into blocks for iBFfixed and the remaining variables
# Z11 = Ztot[iBFfixed, iBFfixed] # scalar
# Z12 = Ztot[iBFfixed, np.arange(Ndes) != iBFfixed] # row vector
# Z21 = Ztot[np.arange(Ndes) != iBFfixed, iBFfixed] # column vector
# Z22 = Ztot[np.ix_(np.arange(Ndes) != iBFfixed, np.arange(Ndes) != iBFfixed)] # matrix
# Vinc1 = Vinc[iBFfixed] # scalar
# Vinc2 = Vinc[np.arange(Ndes) != iBFfixed] # vector

# # Compse reduction to variables excluding iBFfixed
# # Ivec = Tmat @ Ivec2 + Ivec1

# # Tmat is identity with -np.linalg.solve(Z11, Z12) in the iBFfixed row
# Tmat = np.zeros((Ndes, Ndes-1), dtype=complex)
# Tmat[np.arange(Ndes) != iBFfixed, :] = np.eye(Ndes-1)
# Tmat[iBFfixed, :] = -Z12/Z11

# # Ivec1 is zero except for the iBFfixed row, which is np.linalg.solve(Z11, Vinc1)
# Ivec1 = np.zeros(Ndes, dtype=complex)
# Ivec1[iBFfixed] = Vinc1/Z11

# # reduced operators
# Lmatr = Lmat[np.ix_(np.arange(Ndes) != iBFfixed, np.arange(Ndes) != iBFfixed)]
# Ztotr = Z22 - np.outer(Z21, Z12 / Z11)
# Vincr = Vinc2 - Z21 * (Vinc1/Z11)

# # transformed objectives
# Objr = Tmat.conj().T @ Obj @ Tmat
# objr = 2 * Tmat.conj().T @ Obj @ Ivec1 + Tmat.conj().T @ obj
# obj0r = Ivec1.conj().T @ Obj @ Ivec1 + np.real(Ivec1.conj().T @ obj) + obj0

# # verify scattered value of the full plate
# Ivec2 = np.linalg.solve(Ztotr, Vincr)
# Ivec_full = Tmat @ Ivec2 + Ivec1
# Obj_full = np.real(Ivec_full.conj().T @ Obj @ Ivec_full) + np.real(Ivec_full.conj().T @ obj) + obj0
# print(f"Full plate objective: {Obj_full}")

# # Cholesky
# Lchol = np.linalg.cholesky(Lmatr).conj().T
# Mfactor = np.linalg.inv(Lchol)
# Vincf = Mfactor.conj().T @ Vincr
# Ztotf = Mfactor.conj().T @ Ztotr @ Mfactor
# Objf = Mfactor.conj().T @ Objr @ Mfactor
# objf = Mfactor.conj().T @ objr
# obj0f = obj0r

# Ivec2f = np.linalg.solve(Ztotf, Vincf)
# Objf_full = np.real(Ivec2f.conj().T @ Objf @ Ivec2f) + np.real(Ivec2f.conj().T @ objf) + obj0f
# print(f"Full plate objective after transformation: {Objf_full}")




In [ ]:
# Assuming Ndes is OP.BF.nUnknowns and is an odd integer
k = (Ndes - 1) // 2

# 1. Upper block: Identity matrix of size (k + 1)
upper = np.eye(k + 1)

# 2. Lower block: Flipped identity of size k, followed by a column of zeros
# np.fliplr(np.eye(k)) creates the anti-diagonal matrix
lower_left = np.fliplr(np.eye(k))
lower_right = np.zeros((k, 1))
lower = np.hstack([lower_left, lower_right])

# 3. Vertically stack them to create C
C = np.vstack([upper, lower])

# lossy matrix Cholesky factorization
Lchol = np.linalg.cholesky(C.conj().T @Lmat @ C).conj().T

Mfactor = C @ np.linalg.inv(Lchol)
# Mfactor = np.eye(Ndes) # skip transformation for testing purposes
# Mfactor = C

Vincf = Mfactor.conj().T @ Vinc
Ztotf = Mfactor.conj().T @ Ztot @ Mfactor
Z0f = Mfactor.conj().T @ Z0 @ Mfactor
Zmatf = Mfactor.conj().T @ Zmat @ Mfactor
Objf = Mfactor.conj().T @ Obj @ Mfactor
objf = Mfactor.conj().T @ obj
obj0f = obj0

Nfac = Z0f.shape[0]
iVec = np.zeros(Nfac, dtype=complex) # particular solution to satisfy the fixed current constraint (zero for now, can be used to satisfy a fixed current constraint by setting the iBFfixed row to np.linalg.solve(Z11, Vinc1) and the transformation matrix to have -np.linalg.solve(Z11, Z12) in the iBFfixed row)

In [ ]:
# translate QCQP matrices to Dolphindes notation
Umat = 1j * Ztotf.conj() # Dolphindes U matrix
eVec = -1j*Vincf.conj() / 2 # Dolphindes e vector
Bmat = Objf # quadratic objective matrix
bVec = objf # linear objective vector
beta = obj0f # constant objective term

In [ ]:
# preconditioner for projection constraints
G0 = Z0f # Green's function matrix
D = np.diag(np.diag(Z0f)) # Green's matrix diagonal
X = np.linalg.inv(Zmatf) # material admittance matrix
H0 = G0 - D
Y = np.linalg.solve(np.eye(Nfac) + X @ D, X)
iVec = X @ Vincf

S = np.eye(Nfac) + Y @ H0 # preconditioner matrix for projection constraints
q = -Y @ G0 @ iVec

P0 = np.linalg.solve(S, Ztotf)

sVec = np.linalg.solve(S, q)
tVec = sVec + iVec
Vtest = Ztotf @ tVec
print(f"voltage error: {np.max(np.abs(Vtest - Vincf))/np.max(np.abs(Vincf)):.2e}")

print(f"asymmetric error of preconditioner: {np.linalg.norm(S-S.T, ord='fro')/np.linalg.norm(S, ord='fro'):.2e}")
S = (S + S.T) / 2

Ptr0 = -np.real(tVec.conj().T @ Objf @ tVec) + 2*np.real(tVec.conj().T @ objf) + obj0f
print(f"power error without shifting preconditioner: {np.abs(Ptr0 - Pfull)/np.abs(Pfull):.2e}")

obj0f += -np.real(iVec.conj().T  @ Objf @ iVec) + 2*np.real(iVec.conj().T @ objf)
objf += -Objf @ iVec

Ptr = -np.real(sVec.conj().T @ Objf @ sVec) + 2*np.real(sVec.conj().T @ objf) + obj0f
print(f"power error with preconditioner: {np.abs(Ptr - Pfull)/np.abs(Pfull):.2e}")

Umat = 1j * S
eVec = -1j*q / 2
Bmat = Objf
bVec = objf
beta = obj0f

In [ ]:
# set up optimization variables
Nopt = Umat.shape[0] # number of optimization variables (after reduction)

# 1. Use np.column_stack to create a proper 2D NumPy array
# 2. Use np.ones(Nopt) instead of np.ones((Nopt,1))
# Pdiags = np.column_stack([
#     -1j * np.ones((Nopt,1), dtype=complex), 
#     np.eye(Nopt, dtype=complex)
# ])
Pdiags = np.column_stack([
    -1j *np.eye(Nopt, dtype=complex), 
    np.eye(Nopt, dtype=complex)
])

# Now Pdiags is a NumPy array, so .shape and [:,i] will work perfectly
Plist = [sp.diags(Pdiags[:,i]) for i in range(Pdiags.shape[1])]

QCQP = DenseSharedProjQCQP(Bmat, bVec, beta,
                            Umat, eVec,
                             Plist, verbose = 1
                                )


t1 = time.time()
lags_init = np.zeros((Pdiags.shape[1],))
# lags_init[0] = 1
lags_init[0:Nopt] = 1
# New interface for specifying optimization hyperparameters. https://dolphindes.readthedocs.io/en/latest/api/dolphindes.cvxopt.OptimizationHyperparameters.html
opt_params = OptimizationHyperparameters(opttol=1e-15,                  
    gradConverge=True,           
    min_inner_iter=10,             
    max_restart=30,                
    penalty_ratio=1e-3,           
    penalty_reduction=0.01,        
    break_iter_period=5,          
    verbose=1)

# Note: 'newton' is usually faster than BFGS, both are prety fast with so few constraints.
result = QCQP.solve_current_dual_problem(method = 'bfgs', init_lags = lags_init, opt_params = opt_params)
print(f"bound: {result[0]:.4e}, time: {time.time()-t1}s, Lagrange multipliers: {QCQP.current_lags}")

Precomputed 208 A matrices and Fs vectors.
Optimizer initialized with parameters:
opttol: 1e-15
gradConverge: True
min_inner_iter: 10
max_restart: 30
penalty_ratio: 0.001
penalty_reduction: 0.01
break_iter_period: 5
verbose: 1
Starting optimization with x0 = [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Outer iteration 0, penalty_ratio = 0.001, opt_fx = 12.76959893252199
iter_num: 5, prev_fx: inf, opt_fx: 12.